<a href="https://colab.research.google.com/drive/17AVoEYs_sMiMUhqg3Azf-Oo8xY2b2mov?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Skeleton-of-Thought (SoT)

In [ ]:
!pip install -qU google-genai

In [ ]:
from google import genai
import getpass

Get free-tier Google's Gemini API Key here: https://aistudio.google.com/app/apikey

In [ ]:
API_KEY = getpass.getpass("Enter your Google API key: ")

In [ ]:
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-flash-latest"

In [ ]:
import re

In [ ]:
class SkeletonOfThoughtAgent:
    """Answer in two passes: outline first, then expand each point.

    A single generate call writes an answer left to right, so every later
    sentence is conditioned on the earlier ones and the structure emerges by
    accident. Skeleton-of-Thought forces the structure to be a decision:
    produce the skeleton, then expand each point independently. The points do
    not see each other, which is what makes them expandable in parallel.
    """

    def __init__(self, max_points=5):
        self.model = MODEL_NAME
        self.max_points = max_points
        self.skeleton = []

    def build_skeleton(self, question):
        """Pass 1: the outline, and nothing else."""
        prompt = f"""Answer the question below as a skeleton: {self.max_points}
        short points, three to eight words each, no explanation.

        Write one point per line, numbered "1." to "{self.max_points}.".
        Do not write any prose before or after the list.

        Question: {question}"""

        text = client.models.generate_content(
            model=self.model, contents=prompt
        ).text

        points = []
        for line in text.splitlines():
            line = line.strip()
            match = re.match(r"^\s*(\d+)[.)]\s+(.*)$", line)
            if match:
                points.append(match.group(2).strip())

        # A model that ignores the format must not silently produce an empty
        # answer, so fall back to any non-empty line.
        if not points:
            points = [ln.strip(" -*\t") for ln in text.splitlines() if ln.strip()]

        self.skeleton = points[: self.max_points]
        return self.skeleton

    def expand(self, question, index, point):
        """Pass 2: expand one point, with no sight of the others."""
        prompt = f"""Question: {question}

        You are expanding exactly one point of an outline. Write two or three
        sentences on this point only. Do not repeat the point as a heading and
        do not mention the other points.

        Point {index}: {point}"""

        return client.models.generate_content(
            model=self.model, contents=prompt
        ).text.strip()

    def answer(self, question):
        print(f"\nQuestion: {question}\n")

        points = self.build_skeleton(question)
        print("Skeleton:")
        for i, point in enumerate(points, 1):
            print(f"  {i}. {point}")
        print()

        sections = []
        for i, point in enumerate(points, 1):
            body = self.expand(question, i, point)
            sections.append(f"{i}. {point}\n{body}")
            print(f"--- Point {i}: {point} ---")
            print(body + "\n")

        return "\n\n".join(sections)

In [ ]:
# Example 1: an explanatory question
print("=" * 60)
print("EXAMPLE 1: Skeleton first, then expansion")
print("=" * 60)

agent = SkeletonOfThoughtAgent(max_points=4)
agent.answer("Why do neural networks need activation functions?")


# Example 2: the same question answered in one pass, for comparison
print("\n" + "=" * 60)
print("EXAMPLE 2: The same question in a single pass")
print("=" * 60)

single = client.models.generate_content(
    model=MODEL_NAME,
    contents="Why do neural networks need activation functions?",
).text
print(single)

In [ ]:
# What the two passes actually bought you
#
# Read the two outputs above side by side. The interesting comparison is not
# which reads better - it is that the skeleton is inspectable *before* any
# prose exists. You can drop a point, reorder them, or rewrite one and expand
# again, none of which is possible once a single pass has already committed to
# a structure.
#
# The cost is equally concrete: this made 1 + N calls instead of 1.

print(f"Points in the skeleton: {len(agent.skeleton)}")
print(f"API calls made by the skeleton path: {1 + len(agent.skeleton)}")
print("API calls made by the single-pass path: 1")
print()
print("Because the expansions never see each other, they can be issued")
print("concurrently. That is the latency argument for this pattern: the")
print("wall-clock cost approaches one skeleton call plus one expansion,")
print("rather than the sum of every expansion.")